# GeoAI Change Detection Notebook

**STAC querying + lazy COG loading + NDVI change detection + GeoJSON export**

In [ ]:
!pip install -q pystac-client planetary-computer rasterio rioxarray geopandas matplotlib leafmap rasterio shapely

In [ ]:
import pystac_client
import planetary_computer
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import shape
import json

print("✅ Libraries imported")

In [ ]:
# Define small test AOI (fast to run)
bbox = [-100.0, 35.0, -99.5, 35.5]
time_before = "2023-06-01/2023-06-30"
time_after  = "2023-09-01/2023-09-30"

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

search_before = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=time_before, query={"eo:cloud_cover": {"lt": 30}}, limit=3)
search_after  = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=time_after,  query={"eo:cloud_cover": {"lt": 30}}, limit=3)

items_before = list(search_before.items())
items_after  = list(search_after.items())

print(f"Found {len(items_before)} before scenes and {len(items_after)} after scenes")

In [ ]:
def load_band(item, band):
    return rxr.open_rasterio(item.assets[band].href, masked=True).squeeze()

if items_before and items_after:
    red_b = load_band(items_before[0], "B04")
    nir_b = load_band(items_before[0], "B08")
    red_a = load_band(items_after[0], "B04")
    nir_a = load_band(items_after[0], "B08")

    ndvi_before = (nir_b - red_b) / (nir_b + red_b)
    ndvi_after  = (nir_a - red_a) / (nir_a + red_a)
    ndvi_diff   = ndvi_after - ndvi_before

    print("✅ NDVI difference calculated")

## Create Change Polygons and Export GeoJSON

In [ ]:
# Threshold: significant change
threshold = 0.25
change_mask = np.abs(ndvi_diff) > threshold

# Simple vectorization (convert raster change areas to polygons)
from rasterio.features import shapes
import rasterio

with rasterio.open(items_after[0].assets["B04"].href) as src:
    transform = src.transform
    crs = src.crs

shapes_gen = shapes(change_mask.astype(np.uint8), transform=transform)

features = []
for geom, value in shapes_gen:
    if value == 1:  # only changed areas
        features.append({
            "type": "Feature",
            "geometry": geom,
            "properties": {
                "change_type": "Significant Change",
                "ndvi_diff": float(ndvi_diff.where(change_mask).mean().values)
            }
        })

# Save as GeoJSON
geojson = {"type": "FeatureCollection", "features": features}

with open("change_polygons.geojson", "w") as f:
    json.dump(geojson, f)

print(f"✅ Exported {len(features)} change polygons to change_polygons.geojson")
print("Download this file and place it in the root of your repository")

## Key Achievements

- Efficient STAC querying of Sentinel-2 L2A
- Lazy loading of Cloud Optimized GeoTIFFs (COGs)
- NDVI-based automated change detection
- Vectorization of change areas into GeoJSON polygons ready for web map overlay